# UK Turnout Analysis Project

## NB01: Election Results Dataset

This notebook prepares the election-results component of the project dataset.
The House of Commons Library's [constituency-level results for the
2024 UK general election](https://researchbriefings.files.parliament.uk/documents/CBP-10009/HoC-GE2024-results-by-constituency.csv) provide our data source.

### 1. Load election data

In [1]:
from pathlib import Path
import pandas as pd

RAW_DIR = Path("../data/raw")

list(RAW_DIR.iterdir())

[WindowsPath('../data/raw/HoC-GE2024-results-by-constituency.csv')]

In [2]:
csv_path = RAW_DIR / "HoC-GE2024-results-by-constituency.csv"

df = pd.read_csv(csv_path)

print(df.shape)
display(df.head())

(650, 32)


,gitONS ID,ONS region ID,Constituency name,County name,Region name,Country name,Constituency type,Declaration time,Member first name,Member surname,...,Green,SNP,PC,DUP,SF,SDLP,UUP,APNI,All other candidates,Of which other winner
0,W07000081,W92000004,Aberafan Maesteg,NaN,Wales,Wales,County,NaN,Stephen,Kinnock,...,1094,0,4719,0,0,0,0,0,801,0
1,S14000060,S92000003,Aberdeen North,NaN,Scotland,Scotland,Burgh,NaN,Kirsty,Blackman,...,1275,14533,0,0,0,0,0,0,1269,0
2,S14000061,S92000003,Aberdeen South,NaN,Scotland,Scotland,Burgh,NaN,Stephen,Flynn,...,1609,15213,0,0,0,0,0,0,648,0
3,S14000062,S92000003,Aberdeenshire North and Moray East,NaN,Scotland,Scotland,County,NaN,Seamus,Logan,...,0,13455,0,0,0,0,0,0,0,0
4,S14000063,S92000003,Airdrie and Shotts,NaN,Scotland,Scotland,County,NaN,Kenneth,Stevenson,...,0,11324,0,0,0,0,0,0,1079,0


In [3]:
df.columns.tolist()

['gitONS ID',
 'ONS region ID',
 'Constituency name',
 'County name',
 'Region name',
 'Country name',
 'Constituency type',
 'Declaration time',
 'Member first name',
 'Member surname',
 'Member gender',
 'Result',
 'First party',
 'Second party',
 'Electorate',
 'Valid votes',
 'Invalid votes',
 'Majority',
 'Con',
 'Lab',
 'LD',
 'RUK',
 'Green',
 'SNP',
 'PC',
 'DUP',
 'SF',
 'SDLP',
 'UUP',
 'APNI',
 'All other candidates',
 'Of which other winner']

In [4]:
df.dtypes

gitONS ID                    str
ONS region ID                str
Constituency name            str
County name              float64
Region name                  str
Country name                 str
Constituency type            str
Declaration time         float64
Member first name            str
Member surname               str
Member gender                str
Result                       str
First party                  str
Second party                 str
Electorate                 int64
Valid votes                int64
Invalid votes              int64
Majority                   int64
Con                        int64
Lab                        int64
LD                         int64
RUK                        int64
Green                      int64
SNP                        int64
PC                         int64
DUP                        int64
SF                         int64
SDLP                       int64
UUP                        int64
APNI                       int64
All other 

### 2. Validate the raw election data

Before restricting the analysis sample, I check that the reported party vote
totals reconcile exactly with the published number of valid votes in every
constituency.

In [5]:
# Validate source data before subsetting
party_cols = [
    "Con", "Lab", "LD", "RUK", "Green",
    "SNP", "PC", "DUP", "SF", "SDLP",
    "UUP", "APNI", "All other candidates"
]

df["vote_total_check"] = df[party_cols].sum(axis=1)

assert (df["vote_total_check"] == df["Valid votes"]).all()

### 3. Define the analytical sample

The analysis is restricted to England and Wales because Census 2021 data for
both countries are published through the same ONS system and use consistent
definitions. Scotland and Northern Ireland use separate census systems and are
left for a possible later extension.

In [6]:
election = df[df["Country name"].isin(["England", "Wales"])].copy()

election.shape

(575, 33)

In [7]:
print("Rows:", len(election))
print("Unique ONS IDs:", election["ONS ID"].nunique())
print("Missing ONS IDs:", election["ONS ID"].isna().sum())
print("Duplicate ONS IDs:", election["ONS ID"].duplicated().sum())
print(election["Country name"].value_counts())

assert election["ONS ID"].notna().all()
assert election["ONS ID"].is_unique

Rows: 575


KeyError: 'ONS ID'

### 4. Construct voter turnout

Turnout is calculated as all ballots cast, including rejected or invalid
ballots, as a percentage of the registered electorate.

In [ ]:
election["turnout_pct"] = (
    (election["Valid votes"] + election["Invalid votes"])
    / election["Electorate"]
    * 100
)

In [ ]:
election["turnout_pct"].describe()

count    575.000000
mean      60.116025
std        7.010406
min       40.251763
25%       55.005884
50%       60.674673
75%       65.747235
max       75.474101
Name: turnout_pct, dtype: float64

In [ ]:
assert election["turnout_pct"].between(0, 100).all()

### 5. Create the processed election dataset

The final election dataset retains the geographic identifiers and variables
needed for later merging with demographic and socioeconomic data.

In [ ]:
election_clean = election[
    [
        "ONS ID",
        "Constituency name",
        "Region name",
        "Country name",
        "Electorate",
        "Valid votes",
        "Invalid votes",
        "Majority",
        "First party",
        "Second party",
        "turnout_pct",
    ]
].copy()

In [ ]:
election_clean.columns = [
    "constituency_code",
    "constituency_name",
    "region_name",
    "country_name",
    "electorate",
    "valid_votes",
    "invalid_votes",
    "majority",
    "first_party",
    "second_party",
    "turnout_pct",
]

### 6. Export cleaned data

In [ ]:
election_clean.to_csv(
    "../data/processed/election_results_2024.csv",
    index=False
)